# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

In [1]:
import { init, Bool, Arith } from 'z3-solver';
const { Context } = await init();
const ctx = Context("main");

## Auxiliary Functions

The symbolic transition system uses 3 variables.
* `M` is the number of missionaries on the left side of the river,
* `C` is the number of infidels on the left side of the river,
* `B` is the number of boats on the left side of the river.

We define these variables using `ctx.Int.const`.

In [2]:
const M = ctx.Int.const('M');
const C = ctx.Int.const('C');
const B = ctx.Int.const('B');

The function `start` takes three `Z3` variables (integer constants) as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a `Z3` formula (using `ctx.And`) that specifies that everybody is on the western shore.

In [3]:
function start(M: Arith, C: Arith, B: Arith): Bool {
    return ctx.And(M.eq(3), C.eq(3), B.eq(1));
}

In [4]:
start(M, C, B).toString()

(and (= M 3) (= C 3) (= B 1))


The function `goal` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the eastern shore.

In [5]:
function goal(M: Arith, C: Arith, B: Arith): Bool {
    return ctx.And(M.eq(0), C.eq(0), B.eq(0));
}

The function `invariant` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns an array of formulas. The conjunction of these formulas is `True` if there is no problem on either shore of the river. There is no problem if any of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M} = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.

In [6]:
function invariant(M: Arith, C: Arith, B: Arith): Bool[] {
    return [
        ctx.Or(M.eq(0), M.eq(3), M.eq(C)),
        M.ge(0), M.le(3),
        C.ge(0), C.le(3),
        B.ge(0), B.le(1)
    ];
}

In [7]:
for (const constraint of invariant(M, C, B)) {
    console.log(constraint.toString());
}

(or (= M 0) (= M 3) (= M C))
(>= M 0)
(<= M 3)
(>= C 0)
(<= C 3)
(>= B 0)
(<= B 1)


The function `transition` takes 6 arguments:
* `Ma` is the number of missionaries on the eastern shore before the crossing.
* `Ca` is the number of infidels on the eastern shore before the crossing.
* `Ba` is the number of boats on the eastern shore before the crossing. 
* `Mb` is the number of missionaries on the eastern shore after the crossing.
* `Cb` is the number of infidels on the eastern shore after the crossing.
* `Bb` is the number of infidels on the eastern shore after the crossing.

The function returns an array of formulas that is `True` if the missionaries starting on one shore arrive at the opposite shore after the $i^{\textrm{th}}$ crossing. Note that if $i$ is odd, then during the $i^{\textrm{th}}$ crossing the boat travels from the western shore to the eastern shore. If $i$ is even, the boat travels from the eastern shore to the western shore.

In [9]:
function boatOK(Ma: Arith, Ca: Arith, Ba: Arith, Mb: Arith, Cb: Arith, Bb: Arith): Bool[] {
    const diffM = Ma.sub(Mb);
    const diffC = Ca.sub(Cb);
    const totalDiff = diffM.add(diffC);
    return [
        ctx.Implies(
            Ba.eq(1),
            ctx.And(
                totalDiff.ge(1),
                totalDiff.le(2),
                Mb.le(Ma),
                Cb.le(Ca)
            )
        )
    ];
}

In [10]:
const MX = ctx.Int.const('MX');
const CX = ctx.Int.const('CX');
const BX = ctx.Int.const('BX');
const constraints = boatOK(M, C, B, MX, CX, BX);
for (const c of constraints) {
    console.log(c.toString());
}

(let ((a!1 (and (>= (+ (- M MX) (- C CX)) 1)
                (<= (+ (- M MX) (- C CX)) 2)
                (<= MX M)
                (<= CX C))))
  (=> (= B 1) a!1))


In [11]:
function transition(Ma: Arith, Ca: Arith, Ba: Arith, Mb: Arith, Cb: Arith, Bb: Arith): Bool[] { 
    const formulas = [ Bb.eq(Ba.mul(-1).add(1)) ];
    formulas.push(...boatOK(Ma, Ca, Ba, Mb, Cb, Bb));
    formulas.push(...boatOK(Mb, Cb, Bb, Ma, Ca, Ba));
    return formulas;
}

In [12]:
transition(M, C, B, MX, CX, BX).toString();

(= BX (+ (* B (- 1)) 1)),(let ((a!1 (and (>= (+ (- M MX) (- C CX)) 1)
                (<= (+ (- M MX) (- C CX)) 2)
                (<= MX M)
                (<= CX C))))
  (=> (= B 1) a!1)),(let ((a!1 (and (>= (+ (- MX M) (- CX C)) 1)
                (<= (+ (- MX M) (- CX C)) 2)
                (<= M MX)
                (<= C CX))))
  (=> (= BX 1) a!1))


The async function `missionariesCSP` creates a CSP that tries to solve the problem with `n` crossings. It creates a new `Solver` instance, adds all constraints (start, goal, invariants, transitions) and checks for satisfiability.

In [13]:
async function missionariesCSP(n: number): Promise<Record<string, number> | null> {
    const solver = new ctx.Solver();
    const Ms = [], Cs = [], Bs = [];
    for (let i = 0; i <= n; i++) {
        Ms.push(ctx.Int.const(`M${i}`));
        Cs.push(ctx.Int.const(`C${i}`));
        Bs.push(ctx.Int.const(`B${i}`));
    }
    solver.add(start(Ms[0], Cs[0], Bs[0]));
    solver.add(goal(Ms[n], Cs[n], Bs[n]));
    for (let i = 0; i < n; i++) {
        solver.add(...invariant(Ms[i], Cs[i], Bs[i]));
        solver.add(...transition(Ms[i], Cs[i], Bs[i], Ms[i+1], Cs[i+1], Bs[i+1]));
    }
    solver.add(...invariant(Ms[n], Cs[n], Bs[n]));
    const result = await solver.check();
    if (result === 'sat') {
        const model = solver.model();
        const solution: Record<string, number> = {};
        for (let i = 0; i <= n; i++) {
            solution[`M${i}`] = parseInt(model.eval(Ms[i]).toString());
            solution[`C${i}`] = parseInt(model.eval(Cs[i]).toString());
            solution[`B${i}`] = parseInt(model.eval(Bs[i]).toString());
        }
        return solution;
    } else {
        return null;
    }
}

The function `findSolution` computes a solution to the problem of saving the infidels.

In [14]:
async function findSolution(): Promise<{ n: number; solution: Record<string, number>; }> {
    let n = 1;
    while (true) {
        console.log(`Trying n=${n}...`);
        const solution = await missionariesCSP(n);
        if (solution !== null) {
            return { n, solution };
        }
        n += 2;
    }
}

On my desktop computer (2017 iMac with 3.4 GHz Quad-Core Intel i5) it takes about 2 seconds to solve the problem. 

In [15]:
console.time("Missionaries Solving");
const result = await findSolution();
console.timeEnd("Missionaries Solving");
result.solution

Trying n=1...
Trying n=3...
Trying n=5...
Trying n=7...
Trying n=9...
Trying n=11...
Missionaries Solving: 340.087ms
{
  M0: 3,
  C0: 3,
  B0: 1,
  M1: 3,
  C1: 1,
  B1: 0,
  M2: 3,
  C2: 2,
  B2: 1,
  M3: 3,
  C3: 0,
  B3: 0,
  M4: 3,
  C4: 1,
  B4: 1,
  M5: 1,
  C5: 1,
  B5: 0,
  M6: 2,
  C6: 2,
  B6: 1,
  M7: 0,
  C7: 2,
  B7: 0,
  M8: 0,
  C8: 3,
  B8: 1,
  M9: 0,
  C9: 1,
  B9: 0,
  M10: 0,
  C10: 2,
  B10: 1,
  M11: 0,
  C11: 0,
  B11: 0
}


In [16]:
function show_solution(sol: Record<string, number>, n: number) {
    if (!sol) return;
    for (let i = 0; i <= n; i++) {
        const M = sol[`M${i}`];
        const C = sol[`C${i}`];
        const B = sol[`B${i}`];
        const west = '😇'.repeat(M) + '🥷'.repeat(C);
        const east = '😇'.repeat(3 - M) + '🥷'.repeat(3 - C);
        const riverWidth = 28; 
        let line = `${west}${' '.repeat(riverWidth)}${east}`;
        console.log(line);
        if (B === 1 && i < n) {
            const MB = sol[`M${i}`] - sol[`M${i+1}`];
            const CB = sol[`C${i}`] - sol[`C${i+1}`];
            const boatContent = '😇'.repeat(MB) + '🥷'.repeat(CB);
            console.log(' '.repeat(12) + '>>> ' + boatContent + ' >>>');
        } else if (i < n) {
            const MB = sol[`M${i+1}`] - sol[`M${i}`];
            const CB = sol[`C${i+1}`] - sol[`C${i}`];
            const boatContent = '😇'.repeat(MB) + '🥷'.repeat(CB);
            console.log(' '.repeat(12) + '<<< ' + boatContent + ' <<<');
        }
    }
}

In [17]:
show_solution(result.solution, result.n);

😇😇😇🥷🥷🥷                            
            >>> 🥷🥷 >>>
😇😇😇🥷                            🥷🥷
            <<< 🥷 <<<
😇😇😇🥷🥷                            🥷
            >>> 🥷🥷 >>>
😇😇😇                            🥷🥷🥷
            <<< 🥷 <<<
😇😇😇🥷                            🥷🥷
            >>> 😇😇 >>>
😇🥷                            😇😇🥷🥷
            <<< 😇🥷 <<<
😇😇🥷🥷                            😇🥷
            >>> 😇😇 >>>
🥷🥷                            😇😇😇🥷
            <<< 🥷 <<<
🥷🥷🥷                            😇😇😇
            >>> 🥷🥷 >>>
🥷                            😇😇😇🥷🥷
            <<< 🥷 <<<
🥷🥷                            😇😇😇🥷
            >>> 🥷🥷 >>>
                            😇😇😇🥷🥷🥷
